In [ ]:
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Image, display

import torch
from ultralytics import YOLO

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
DATA_ROOT = Path("data/HRPlanes")

train_img_dir = DATA_ROOT / "images" / "train"
val_img_dir = DATA_ROOT / "images" / "val"
test_img_dir = DATA_ROOT / "images" / "test"

train_label_dir = DATA_ROOT / "labels" / "train"
val_label_dir = DATA_ROOT / "labels" / "val"
test_label_dir = DATA_ROOT / "labels" / "test"

print("train:", len(list(train_img_dir.glob("*"))), len(list(train_label_dir.glob("*"))))
print("val  :", len(list(val_img_dir.glob("*"))), len(list(val_label_dir.glob("*"))))
print("test :", len(list(test_img_dir.glob("*"))), len(list(test_label_dir.glob("*"))))

In [ ]:
DATA_YAML = Path("hrplanes.yaml")

print(DATA_YAML.read_text())

In [ ]:
MODEL_NAME = "yolov8s.pt"

IMG_SIZE = 960
BATCH_SIZE = 16
NUM_EPOCHS = 50
NUM_WORKERS = 8

LR = 1e-3
TARGET_MAP50 = 0.70

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
sample_images = sorted(train_img_dir.glob("*"))[:6]

plt.figure(figsize=(12, 6))

for i, img_path in enumerate(sample_images):
    img = PILImage.open(img_path).convert("RGB")
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=NUM_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    optimizer="AdamW",
    lr0=LR,
    workers=NUM_WORKERS,
    seed=SEED,
    project="runs/lab3",
    name="airplane_yolov8s",
    exist_ok=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    plots=True
)

run_dir = Path(train_results.save_dir)
print("run_dir:", run_dir)

In [ ]:
history_path = run_dir / "results.csv"
history_df = pd.read_csv(history_path)
history_df.columns = history_df.columns.str.strip()

print(history_df.tail())
print(history_df.columns.tolist())

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(history_df["epoch"], history_df["train/box_loss"], label="train_box_loss")
plt.plot(history_df["epoch"], history_df["val/box_loss"], label="val_box_loss")
plt.legend()
plt.title("Box loss")

plt.subplot(1, 3, 2)
plt.plot(history_df["epoch"], history_df["metrics/mAP50(B)"], label="mAP50")
plt.axhline(TARGET_MAP50, linestyle="--", label="target")
plt.legend()
plt.title("mAP50")

plt.subplot(1, 3, 3)
plt.plot(history_df["epoch"], history_df["metrics/precision(B)"], label="precision")
plt.plot(history_df["epoch"], history_df["metrics/recall(B)"], label="recall")
plt.legend()
plt.title("Precision / Recall")

plt.tight_layout()
plt.show()

In [ ]:
best_weights = run_dir / "weights" / "best.pt"

best_model = YOLO(str(best_weights))

metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=NUM_WORKERS,
    project="runs/lab3",
    name="airplane_yolov8s_val",
    plots=True
)

In [ ]:
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

print(f"precision = {precision:.4f}")
print(f"recall    = {recall:.4f}")
print(f"f1        = {f1:.4f}")
print(f"mAP50     = {map50:.4f}")
print(f"mAP50-95  = {map50_95:.4f}")

if map50 > TARGET_MAP50:
    print("target metric passed")
else:
    print("target metric failed")

In [ ]:
report = {
    "target_metric": "mAP50",
    "target_value": TARGET_MAP50,
    "passed": map50 > TARGET_MAP50,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "mAP50": map50,
    "mAP50-95": map50_95,
    "best_weights": str(best_weights),
}

report_path = run_dir / "lab3_metrics.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(report_path)
print(json.dumps(report, indent=2))

In [ ]:
test_images = sorted(test_img_dir.glob("*"))[:5]

pred_results = best_model.predict(
    source=[str(p) for p in test_images],
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    project="runs/lab3",
    name="predictions",
    exist_ok=True,
    save=True
)

pred_dir = Path("runs/lab3/predictions")
print(pred_dir)

In [ ]:
for img_path in sorted(pred_dir.glob("*"))[:5]:
    display(Image(filename=str(img_path)))